# Reproducibility: Calibration Behaviour of Diffusion Models Under Distribution Shift

Arati Dipak Sutar - Matriculation 80433940

This notebook reproduces every numerical result in the thesis proposal from saved
posterior samples. It requires no GPU: sampling was performed separately and the
samples are attached as inputs.

## Required inputs

Attach via **Add Input**:

| Input | Contents | Produces |
|---|---|---|
| `diffusion-model` (notebook output) | `F_*`, `NF_*`, downscale 4 | Sections 6.2, 6.3, 6.5, 6.6, 6.8 |
| `scale16-results` (dataset) or `re-run-scale-16` (notebook output) | `F16_*`, `NF16_*`, downscale 16 | Sections 6.1, 6.3, 6.5, 6.8 |
| `notebookdf870a8e49` (notebook output) | `G_*`, guidance sweep | Section 6.7 |

Set **Accelerator: None**.

## Sampling code

Samples were generated with unmodified Diffusion Posterior Sampling
(https://github.com/DPS2022/diffusion-posterior-sampling), checkpoint `ffhq_10m.pt`,
via `sample_condition.py`. Only the task config was edited: the data root, the
`scale_factor`, and the conditioning `scale`. The sampling loop used is reproduced
in the appendix cell at the end.

In [1]:
import numpy as np, glob, os
from PIL import Image

rng = np.random.default_rng(0)
K = 6                      # samples per image
NOMINAL = 0.90             # target coverage

def load(p):
    return np.array(Image.open(p).convert('RGB')).astype(float)

def find_base(pattern):
    """locate the attached input containing a given folder pattern"""
    for root in ['/kaggle/input/notebooks/*/*', '/kaggle/input/datasets/*/*',
                 '/kaggle/input/*']:
        for p in glob.glob(root):
            if glob.glob(f'{p}/{pattern}'):
                return p
    return None

BASE4  = find_base('F_00000')
BASE16 = find_base('F16_00000')
BASEG  = find_base('G_F0.3_00000')

print('downscale 4  :', BASE4)
print('downscale 16 :', BASE16)
print('guidance     :', BASEG)

downscale 4  : /kaggle/input/notebooks/aratisutar/diffusion-model
downscale 16 : /kaggle/input/notebooks/aratisutar/re-run-scale-16
guidance     : /kaggle/input/notebooks/aratisutar/notebookdf870a8e49


## Data loading

In [2]:
def load_group(base, prefix, n_img, k=K):
    """returns S (n, k, 256, 256, 3) and T (n, 256, 256, 3)"""
    S, T = [], []
    for i in range(n_img):
        fs = sorted(glob.glob(f'{base}/{prefix}_{i:05d}/*.png'))[:k]
        if len(fs) < k:
            continue
        S.append(np.stack([load(f) for f in fs]))
        T.append(load(f'{base}/truth_{prefix}_{i:05d}.png'))
    if not S:
        raise RuntimeError(f'no images found for {base}/{prefix}_*')
    return np.stack(S), np.stack(T)

S_f4,  T_f4  = load_group(BASE4,  'F',    10)
S_n4,  T_n4  = load_group(BASE4,  'NF',    9)
S_f16, T_f16 = load_group(BASE16, 'F16',  10)
S_n16, T_n16 = load_group(BASE16, 'NF16',  9)

for name, S in [('faces d4', S_f4), ('nonfaces d4', S_n4),
                ('faces d16', S_f16), ('nonfaces d16', S_n16)]:
    print(f'{name:14s} {S.shape}')

faces d4       (10, 6, 256, 256, 3)
nonfaces d4    (9, 6, 256, 256, 3)
faces d16      (10, 6, 256, 256, 3)
nonfaces d16   (9, 6, 256, 256, 3)


## Metric 1: spread-to-error ratio

For k samples of one input with ground truth T:

    spread = mean over all C(k,2) pairs of  mean|S_i - S_j|
    error  = mean over all k samples  of    mean|S_i - T|
    ratio  = spread / error

Reported as the mean over images of the per-image ratio.

In [3]:
def ratio_per_image(S, T):
    out = []
    for S1, T1 in zip(S, T):
        k = len(S1)
        sp = np.mean([np.abs(S1[a]-S1[b]).mean()
                      for a in range(k) for b in range(a+1, k)])
        er = np.mean([np.abs(s-T1).mean() for s in S1])
        out.append((sp, er, sp/er))
    return np.array(out)

print('=== Section 6.1 and 6.2: spread-to-error ratio ===\n')
print(f"{'group':16s} {'spread':>8s} {'error':>8s} {'ratio':>8s} {'sd':>7s} {'n':>4s}")
R = {}
for name, S, T in [('faces d4', S_f4, T_f4), ('nonfaces d4', S_n4, T_n4),
                   ('faces d16', S_f16, T_f16), ('nonfaces d16', S_n16, T_n16)]:
    r = ratio_per_image(S, T); R[name] = r
    print(f'{name:16s} {r[:,0].mean():8.2f} {r[:,1].mean():8.2f} '
          f'{r[:,2].mean():8.3f} {r[:,2].std():7.3f} {len(r):4d}')

print('\n=== Section 6.3: interaction ===\n')
g4  = R['faces d4'][:,2].mean()  - R['nonfaces d4'][:,2].mean()
g16 = R['faces d16'][:,2].mean() - R['nonfaces d16'][:,2].mean()
print(f'ratio gap, downscale  4 : {g4:+.3f}')
print(f'ratio gap, downscale 16 : {g16:+.3f}')

=== Section 6.1 and 6.2: spread-to-error ratio ===

group              spread    error    ratio      sd    n
faces d4             9.75    11.39    0.864   0.072   10
nonfaces d4          7.84    11.01    0.747   0.131    9
faces d16           20.17    20.53    0.984   0.057   10
nonfaces d16        18.33    20.64    0.900   0.094    9

=== Section 6.3: interaction ===

ratio gap, downscale  4 : +0.117
ratio gap, downscale 16 : +0.083


## Metric 2: conformal coverage

Per pixel, the interval is mean(S) +/- lambda * sd(S). Lambda is calibrated on the
in-distribution group so that empirical coverage equals the nominal level, then
applied unchanged to the shifted group.

In [4]:
def calibrate_and_report(Sf, Tf, Sn, Tn, tag, nominal=NOMINAL):
    # compute per-pixel statistics once, in float32
    mu_f = Sf.mean(axis=1).astype(np.float32)
    sd_f = (Sf.std(axis=1) + 1e-8).astype(np.float32)
    dev_f = np.abs(Tf.astype(np.float32) - mu_f) / sd_f   # deviation in sd units

    mu_n = Sn.mean(axis=1).astype(np.float32)
    sd_n = (Sn.std(axis=1) + 1e-8).astype(np.float32)
    dev_n = np.abs(Tn.astype(np.float32) - mu_n) / sd_n

    # lambda giving the nominal coverage is just the quantile of dev_f
    lam = float(np.quantile(dev_f, nominal))

    cf = float((dev_f <= lam).mean())
    cn = float((dev_n <= lam).mean())
    print(f'{tag:14s} lambda {lam:5.2f}   faces {cf:.3f}   '
          f'nonfaces {cn:.3f}   gap {cf-cn:+.3f}')
    return lam, dev_f, dev_n

print('=== Section 6.5: pooled per-pixel coverage ===\n')
LAM = {}
DEV = {}
for tag, Sf, Tf, Sn, Tn in [('downscale 4',  S_f4,  T_f4,  S_n4,  T_n4),
                            ('downscale 16', S_f16, T_f16, S_n16, T_n16)]:
    lam, df, dn = calibrate_and_report(Sf, Tf, Sn, Tn, tag)
    LAM[tag] = lam
    DEV[tag] = (df, dn)

=== Section 6.5: pooled per-pixel coverage ===

downscale 4    lambda  3.12   faces 0.900   nonfaces 0.869   gap +0.031
downscale 16   lambda  2.43   faces 0.900   nonfaces 0.868   gap +0.032


## Section 6.6: gradient-restricted coverage, with mask controls

Three masks of identical size (10 per cent of pixels) separate the effect of the
gradient selection from the effect of the reduced pixel count.

In [5]:
def gradient_masks(T_batch, frac=0.10):
    """returns per-image boolean masks: top, bottom, random"""
    tops, bots, rands = [], [], []
    for T1 in T_batch:
        g = np.abs(np.gradient(T1.mean(axis=2))[0]) + \
            np.abs(np.gradient(T1.mean(axis=2))[1])
        hi, lo = np.quantile(g, 1-frac), np.quantile(g, frac)
        r = np.zeros(g.size, dtype=bool)
        r[rng.choice(g.size, int(g.size*frac), replace=False)] = True
        tops.append(np.repeat((g >= hi)[:,:,None], 3, axis=2))
        bots.append(np.repeat((g <= lo)[:,:,None], 3, axis=2))
        rands.append(np.repeat(r.reshape(g.shape)[:,:,None], 3, axis=2))
    return tops, bots, rands

def masked_cov(dev, T_batch, lam, kind, frac=0.10):
    tops, bots, rands = gradient_masks(T_batch, frac)
    sel = {'top': tops, 'bottom': bots, 'random': rands}
    out = []
    for i, d in enumerate(dev):
        if kind == 'all':
            out.append((d <= lam).mean())
        else:
            m = sel[kind][i]
            out.append((d[m] <= lam).mean())
    return np.array(out)

lam4 = LAM['downscale 4']
dev_f4, dev_n4 = DEV['downscale 4']
print(f'lambda calibrated on faces (downscale 4) = {lam4:.2f}\n')

print('--- threshold sweep, top-k per cent gradient ---\n')
print(f"{'mask':>10s} {'faces':>8s} {'nonfaces':>9s} {'gap':>8s}")
for frac in [0.50, 0.30, 0.20, 0.10, 0.05, 0.02]:
    cf = masked_cov(dev_f4, T_f4, lam4, 'top', frac)
    cn = masked_cov(dev_n4, T_n4, lam4, 'top', frac)
    print(f'{frac*100:9.0f}% {cf.mean():8.3f} {cn.mean():9.3f} '
          f'{cf.mean()-cn.mean():+8.3f}')

print('\n--- mask controls, all 10 per cent of pixels ---\n')
print(f"{'mask':>10s} {'faces':>8s} {'nonfaces':>9s} {'gap':>8s}")
store = {}
for kind in ['all', 'top', 'bottom', 'random']:
    cf = masked_cov(dev_f4, T_f4, lam4, kind)
    cn = masked_cov(dev_n4, T_n4, lam4, kind)
    store[kind] = (cf, cn)
    print(f'{kind:>10s} {cf.mean():8.3f} {cn.mean():9.3f} '
          f'{cf.mean()-cn.mean():+8.3f}')

print('\n--- bootstrap CI for the top-10 per cent gap ---\n')
cf, cn = store['top']
boot = np.array([rng.choice(cf, len(cf), replace=True).mean()
                 - rng.choice(cn, len(cn), replace=True).mean()
                 for _ in range(5000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'gap {cf.mean()-cn.mean():.3f}   95% CI [{lo:.3f}, {hi:.3f}]   '
      f'(5000 resamples over images)')

lambda calibrated on faces (downscale 4) = 3.12

--- threshold sweep, top-k per cent gradient ---

      mask    faces  nonfaces      gap
       50%    0.893     0.825   +0.068
       30%    0.886     0.786   +0.101
       20%    0.883     0.760   +0.123
       10%    0.881     0.732   +0.150
        5%    0.881     0.727   +0.154
        2%    0.879     0.726   +0.152

--- mask controls, all 10 per cent of pixels ---

      mask    faces  nonfaces      gap
       all    0.900     0.869   +0.031
       top    0.881     0.732   +0.150
    bottom    0.901     0.924   -0.023
    random    0.900     0.871   +0.029

--- bootstrap CI for the top-10 per cent gap ---

gap 0.150   95% CI [0.094, 0.221]   (5000 resamples over images)


## Section 6.7: guidance-scale control

The conditioning scale directly controls sample spread and is therefore a plausible
confound. Four values were tested on 5 faces and 5 non-faces.

In [6]:
def ratio_guidance(base, tag, n_img=5, k=K):
    out = []
    for i in range(n_img):
        fs = sorted(glob.glob(f'{base}/G_{tag}_{i:05d}/*.png'))[:k]
        if len(fs) < k:
            continue
        S = np.stack([load(f) for f in fs])
        T = load(f'{base}/truth_{tag}_{i:05d}.png')
        sp = np.mean([np.abs(S[a]-S[b]).mean()
                      for a in range(k) for b in range(a+1, k)])
        er = np.mean([np.abs(s-T).mean() for s in S])
        out.append((sp, er, sp/er))
    return np.array(out) if out else None

print('=== Section 6.7: guidance-scale sweep ===\n')
print(f"{'grp':4s} {'gs':5s} {'spread':>8s} {'error':>8s} {'ratio':>8s} {'sd':>7s}")
G = {}
for grp in ['F', 'NF']:
    for gs in ['0.1', '0.3', '0.6', '1.0']:
        r = ratio_guidance(BASEG, f'{grp}{gs}')
        if r is None:
            print(f'{grp:4s} {gs:5s}  missing'); continue
        G[(grp, gs)] = r
        print(f'{grp:4s} {gs:5s} {r[:,0].mean():8.2f} {r[:,1].mean():8.2f} '
              f'{r[:,2].mean():8.3f} {r[:,2].std():7.3f}')

print('\ngap by guidance scale:')
for gs in ['0.1', '0.3', '0.6', '1.0']:
    if ('F', gs) in G and ('NF', gs) in G:
        print(f"  gs={gs}:  {G[('F',gs)][:,2].mean() - G[('NF',gs)][:,2].mean():+.3f}")

print('\nreconstruction error (validity of the operating point):')
for gs in ['0.1', '0.3', '0.6', '1.0']:
    if ('F', gs) in G and ('NF', gs) in G:
        print(f"  gs={gs}:  faces {G[('F',gs)][:,1].mean():6.2f}   "
              f"nonfaces {G[('NF',gs)][:,1].mean():6.2f}")

=== Section 6.7: guidance-scale sweep ===

grp  gs      spread    error    ratio      sd
F    0.1      17.10    19.50    0.894   0.087
F    0.3       9.58    10.97    0.884   0.055
F    0.6       7.21     8.37    0.869   0.063
F    1.0       6.22     7.09    0.889   0.077
NF   0.1      14.75    17.50    0.867   0.078
NF   0.3       7.01    10.74    0.709   0.157
NF   0.6       5.39     8.87    0.664   0.147
NF   1.0       4.82     7.61    0.712   0.162

gap by guidance scale:
  gs=0.1:  +0.027
  gs=0.3:  +0.175
  gs=0.6:  +0.205
  gs=1.0:  +0.177

reconstruction error (validity of the operating point):
  gs=0.1:  faces  19.50   nonfaces  17.50
  gs=0.3:  faces  10.97   nonfaces  10.74
  gs=0.6:  faces   8.37   nonfaces   8.87
  gs=1.0:  faces   7.09   nonfaces   7.61


## Section 6.8: frequency decomposition

Spread and error are decomposed into radial spatial-frequency bands via the 2D FFT,
and the ratio is computed per band. Bands are in cycles per image width. Power is
converted to amplitude by taking the square root, so the values are comparable to
the spatial-domain ratio.

In [7]:
N = 256
fy, fx = np.meshgrid(np.fft.fftfreq(N)*N, np.fft.fftfreq(N)*N, indexing='ij')
radius = np.sqrt(fy**2 + fx**2)
EDGES = np.array([0, 8, 16, 24, 32, 48, 64, 96, 128])
MASKS = [(radius >= EDGES[b]) & (radius < EDGES[b+1])
         for b in range(len(EDGES)-1)]

def band_power(D):
    F = np.abs(np.fft.fft2(D))**2
    return np.array([F[m].mean() for m in MASKS])

def band_ratio(S, T, k=K):
    out = []
    for S1, T1 in zip(S, T):
        Sg = S1.mean(axis=-1)          # greyscale
        Tg = T1.mean(axis=-1)
        sp = np.mean([band_power(Sg[a]-Sg[b])
                      for a in range(k) for b in range(a+1, k)], axis=0)
        er = np.mean([band_power(s-Tg) for s in Sg], axis=0)
        out.append(np.sqrt(sp/er))
    return np.array(out)

bf4,  bn4  = band_ratio(S_f4,  T_f4),  band_ratio(S_n4,  T_n4)
bf16, bn16 = band_ratio(S_f16, T_f16), band_ratio(S_n16, T_n16)

print('=== Section 6.8: frequency decomposition ===\n')
print(f"{'band':>10s} | {'faces':>7s} {'non-f':>7s} {'gap':>8s} | "
      f"{'faces':>7s} {'non-f':>7s} {'gap':>8s}")
print(f"{'(cyc/img)':>10s} | {'--- downscale 4 ---':^24s} | "
      f"{'--- downscale 16 ---':^24s}")
print('-'*72)
for b in range(len(MASKS)):
    print(f'{EDGES[b]:4.0f}-{EDGES[b+1]:<5.0f} | '
          f'{bf4[:,b].mean():7.3f} {bn4[:,b].mean():7.3f} '
          f'{bf4[:,b].mean()-bn4[:,b].mean():+8.3f} | '
          f'{bf16[:,b].mean():7.3f} {bn16[:,b].mean():7.3f} '
          f'{bf16[:,b].mean()-bn16[:,b].mean():+8.3f}')

=== Section 6.8: frequency decomposition ===

      band |   faces   non-f      gap |   faces   non-f      gap
 (cyc/img) |   --- downscale 4 ---    |   --- downscale 16 ---  
------------------------------------------------------------------------
   0-8     |   1.022   1.051   -0.028 |   1.118   1.081   +0.037
   8-16    |   0.996   0.904   +0.091 |   0.931   0.869   +0.063
  16-24    |   0.884   0.720   +0.164 |   0.917   0.846   +0.071
  24-32    |   0.837   0.608   +0.229 |   0.906   0.812   +0.094
  32-48    |   0.812   0.554   +0.258 |   0.882   0.793   +0.089
  48-64    |   0.786   0.462   +0.325 |   0.878   0.727   +0.151
  64-96    |   0.773   0.382   +0.391 |   0.878   0.660   +0.218
  96-128   |   0.832   0.363   +0.469 |   0.948   0.672   +0.276


## Convergence check

The choice of 6 samples per image rests on the ratio stabilising by 4-6 samples.
This recomputes the ratio using the first j samples only.

Note: this check applies to the ratio, not to coverage. Coverage estimated from 6
samples underestimates the true interval width. This is stated as limitation 4.

In [8]:
print('=== convergence of the ratio in the number of samples ===\n')
print(f"{'j':>3s} {'faces d4':>10s} {'nonfaces d4':>12s}")
for j in [2, 3, 4, 5, 6]:
    rf = ratio_per_image(S_f4[:, :j], T_f4)[:, 2].mean()
    rn = ratio_per_image(S_n4[:, :j], T_n4)[:, 2].mean()
    print(f'{j:3d} {rf:10.3f} {rn:12.3f}')

=== convergence of the ratio in the number of samples ===

  j   faces d4  nonfaces d4
  2      0.837        0.801
  3      0.859        0.766
  4      0.871        0.759
  5      0.864        0.746
  6      0.864        0.747


## Appendix: the sampling loop

Reproduced for reference. Requires a GPU and the DPS repository. Config files were
generated by string substitution on `configs/super_resolution_config.yaml`, editing
only the data root, `scale_factor`, and the conditioning `scale`.

```python
import subprocess, shutil, os

os.chdir('/kaggle/working/dps')
for prefix, cfg, n in [('F', 'configs/f10.yaml', 10),
                       ('NF', 'configs/nf9.yaml', 9)]:
    imgs = [f'{i:05d}' for i in range(n)]
    for img in imgs:
        os.makedirs(f'/kaggle/working/{prefix}_{img}', exist_ok=True)
    for rep in range(6):
        # resume-safe: skip any repetition already on disk
        if all(os.path.exists(f'/kaggle/working/{prefix}_{i}/{rep:02d}.png')
               for i in imgs):
            continue
        subprocess.run(['python', 'sample_condition.py',
            '--model_config=configs/model_config.yaml',
            '--diffusion_config=configs/diffusion_config.yaml',
            f'--task_config={cfg}'], check=True, capture_output=True)
        for img in imgs:
            shutil.copy(f'results/super_resolution/recon/{img}.png',
                        f'/kaggle/working/{prefix}_{img}/{rep:02d}.png')
            shutil.copy(f'results/super_resolution/label/{img}.png',
                        f'/kaggle/working/truth_{prefix}_{img}.png')
```

**Ground truth.** Taken from the pipeline's own `label` output rather than the input
file. DPS applies internal normalisation; comparing model output against an
unnormalised source image introduces approximately 9.5 intensity units of spurious
error.

**Dependency note.** The repository pins PyTorch 1.11; the code runs unmodified on
PyTorch 2.10 after providing a stub for the `motionblur` module, which is imported
at the top of `measurements.py` but used only by the motion-deblurring task.